In [ ]:
%pylab inline
import eucare as ec
import torch
from tqdm.auto import tqdm
from collections import defaultdict
from einops import asnumpy
from eucare.conway import alternating_flagstone_graph, dual_graph
from eucare.example_graphs import kised_soccer_ball
from eucare.redering import inset_poly
from eucare.overlap import fold_wireframe, fold_complete, color_creases
from eucare.overlap import CREASE_ASSIGNMENT, MOUNTAIN, VALLEY
from eucare.cutting import cut_out_poly
render_settings = dict(line_width=0.02, render_faces=True, face_inset=0, render_vertices=False)
render_settings_big = dict(line_width=0.02, render_faces=False, render_vertices=False, height=2000)

In [ ]:
def reverse_dict(d):
    result = {value: key for key, value in d.items()}
    assert len(result) == len(d), f'Duplicate values in {d}'
    return result

In [ ]:
import networkx as nx

def central_face(G):
    fs = list(G.faces)
    return fs[np.argmin([np.linalg.norm(f.midpoint()) for f in fs])]

def central_vertex(G):
    vs = list(G.vertices)
    return vs[np.argmin([np.linalg.norm(v['pos']) for v in vs])]

# G = from_tiles(eu.example_tilesets.t_4_6_12(), rings=3)
G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(6), rings=5) #6
# G = from_tiles(eu.example_tilesets.platonic(4), rings=9, vertex_based=False)


# remove the central hexagon
# G = ec.conway.kis_graph()(G, faces=[central_face(G)])
# v = central_vertex(G)
# [G.delete_edge(e.nex) for e in v.outgoing_iter()]
# [G.join_vertex(v) for v in list(G.vertices) if v.order() == 2 and not v.on_border()]

#G = conway.dual_graph()(G)
# remove edges crossing the positive x-axis
es = list(G.halfedges)
eps = 1e-6
es = [e for e in es if e.orig['pos'][1] > eps and e.dest['pos'][1] < 0 and e.orig['pos'][0] < 0]
es = sorted(es, key=lambda e: e.orig['pos'][0])
# [G.delete_edge(e) for e in es]

G.show(**render_settings)
ps, vs = G.get_position_view()
#ps[:, 1] *= 1.5
def rot_mat(alpha):
    return np.array([[np.cos(alpha), np.sin(alpha)],[-np.sin(alpha), np.cos(alpha)]])
#ps[:] = ps @ rot_mat(np.pi/12)
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k -= np.mean(k)

k = k**1.2 # hex to pentagon
# k = k**1.5 # hex to square
# k = k**2 # hex to triangle
# k = k**1.333333333333 # square to triangle

k = np.stack([k.real, k.imag], axis=-1)
ps[:] = k
#ps[:] = [p @ rot_mat(np.linalg.norm(p) / 10) for p in ps]

G.recompute_lengths_and_angles()

from eucare.conversions import EHEG_from_nx

def remove_duplicates(G, eps=1e-6, exclude_edges=()):
    vs = list(G.vertices)
    pos = np.stack([v['pos'] for v in vs])

    dists = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    dists[np.eye(len(vs), dtype=bool)] = np.inf

    closest_points = np.argmin(dists, axis=0)
    min_dists = dists[np.arange(len(pos)), closest_points]
    node_mapping = {i: i if (i<j or d > eps) else j for i, (d, j) in enumerate(zip(min_dists, closest_points))}

    v_index = {v: node_mapping[i] for i, v in enumerate(vs)}

    nxG = nx.Graph()
    nxG.add_nodes_from(v_index.values())
    nx_positions = {i: pos[i] for i in node_mapping.values()}
    nxG.add_edges_from([(v_index[e.orig], v_index[e.dest]) for e in G.halfedges 
                        if e not in set(exclude_edges).union({e.rev for e in exclude_edges})])

    G2 = EHEG_from_nx(nxG, nx_positions)
    G2.recompute_lengths_and_angles()
    return G2

G = remove_duplicates(G, exclude_edges=es)

# G.show(**render_settings)
G = ec.conway.dual_graph()(G)

# G = conway.dual_graph()(G)
G.show(**render_settings)


In [ ]:
from eucare.example_graphs import complete_closest_vertices
G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_ambo(5, 3), rings=1)
complete_closest_vertices(G)
G = ec.conway.kis_graph()(G, faces=[f for f in G.faces if f.order() == 5], delete_on_border=False)
G = ec.conway.goldberg2_graph()(G, delete_on_border=False)
G.show()

In [ ]:
# # Hyperbolic squares

# G = ec.example_graphs.from_tiles(ec.example_graphs.curved_platonic(7, 3), 0)
# # h = next(h for h in G.halfedges 
# #          if h.on_border() and h.orig['pos'].real < 0 and h.orig['pos'].imag < 0 and h.dest['pos'].imag > 0)
# # G.execute_edge_instruction(h)

# # geom = G.geometry
# # geom.center_of_mass
# # translate = geom.translation(geom.center_of_mass(np.array([h.orig['pos'], h.dest['pos']])), geom.origin())
# # rotate = geom.rotation(0, np.pi/4)
# # for v in G.vertices:
# #     v['pos'] = rotate(translate(v['pos']))
    
# G.show()
# G = ec.example_graphs.hyperbolic_square_graph(min_length=0.03, G=G)
# G.show(**render_settings)

In [ ]:
from eucare.io import load_graph
# G = load_graph('test_save.heg')
# G = ec.io.load_graph('graphs/hyperbolic_annulus_7_smooth.heg')
# G = ec.io.load_graph('graphs/pent_3.heg')
# G = ec.io.load_graph('graphs/delaunay_1.heg')
G = ec.io.load_graph('graphs/circlepack/octagon.heg')
# G = ec.io.load_graph('graphs/circlepack/tri_spiral.heg')
# G = ec.io.load_graph('graphs/circlepack/tri_spiral_off4.heg')
# G = ec.io.load_graph('graphs/circlepack/mcqueens_1.heg')
# G = ec.io.load_graph('graphs/circlepack/gyro.heg')
# G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(3), rings=3)

def pseudo_incenter(f):
    """
    If f has an incenter, this is it.
    Otherwise, this is the mean of all the incenters of triangles with corners being a set of adjacent corners of f. 
    """
    ps = np.array([v['pos'] for v in f.vertex_iter()])
    lengths = np.linalg.norm(np.roll(ps, 1, axis=0) - np.roll(ps, 2, axis=0), axis=-1)
    incenter = np.sum(lengths[:, None] * ps, axis=0) / np.sum(lengths)
    return incenter
    
for f in list(G.faces):
    f['midpoint'] = pseudo_incenter(f)
    
G.show()

In [ ]:
len(G.vertices)

In [ ]:
G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(3), 5)
G.show()

In [ ]:
m = np.mean([v['pos'] for v in G.vertices], axis=0)
plt.hist([np.linalg.norm(v['pos'] - m) for v in G.vertices])
G.delete_subset([v for v in G.vertices if np.linalg.norm(v['pos']-m) > 0.26])
G.delete_subset([f for f in G.faces if f.order() > 3])
G.delete_subset([v for v in G.vertices if v.on_border() and v.order() < 3])
G.delete_subset([v for v in G.vertices if v.on_border() and v.order() < 3])
G.delete_subset([v for v in G.vertices if v.on_border() and v.order() < 3])
G.show()

In [ ]:
# make border a circle

def geometric_mean(arr):
    assert len(arr.shape) == 1, f'{arr.shape}'
    return np.prod(arr**(1/len(arr)))

vs = [v for v in G.vertices if v.on_border()]
ps = np.stack([v['pos'] for v in vs])
m = np.mean(ps, axis=0)
ps -= m
norms = linalg.norm(ps, axis=-1)
radius = geometric_mean(norms)
ps = ps/norms[:, None] * radius + m
for v, p in zip(vs, ps):
    v['pos'] = p
    
G.show()

In [ ]:
# G = kised_soccer_ball()
# G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_zip(7, 3), rings=5) # 6
# G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_platonic(7, 3), rings=2)
# G = ec.conway.dual_graph()(G)

for obj in G.faces.union(G.vertices):
    if 'pre_conway' in obj:
        del obj['pre_conway']
G.convert_to_euclidean()
# G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(3), rings=1)
p, _ = G.get_position_view()
p /= np.std(p - np.mean(p, axis=0, keepdims=True))
p *= 5

for f in list(G.faces):
    f['midpoint'] = pseudo_incenter(f)

CP, (v_map, e_map, f_map) = G.copy(return_mappings=True)
v_map_rev, e_map_rev, f_map_rev = [reverse_dict(m) for m in (v_map, e_map, f_map)]
CP = alternating_flagstone_graph(t=0.5)(CP)

# CP = dual_graph()(G.copy())

G.show(**render_settings)

# variables are (for now) a rotation and an offset for every face in the original graph.
from eucare.half import Face, Vertex

fs = [f for f in CP.faces if isinstance(f.attributes.get('pre_conway', None), Face)]
face_coords = [torch.tensor(numpy.stack([v['pos'] for v in f.vertex_iter()])) for f in fs]

# dicts to look up corresponding faces and vertices in CP
f_lookup = {f_map_rev[f['pre_conway']]: f for f in fs}
f0_lookup = reverse_dict(f_lookup) 
f0s = [f0_lookup[f] for f in fs]

vstars = [v for v in CP.vertices if isinstance(v.attributes.get('pre_conway', None), Vertex)]
vstar_lookup = {v_map_rev[v['pre_conway']]: v for v in vstars}
vstar0_lookup = reverse_dict(vstar_lookup)
vstar0s = [vstar0_lookup[v] for v in vstars]

vs = []
vf0_lookup = dict()
vstar_groups = defaultdict(set)
for f0, f in f_lookup.items():
    for v0 in f0.vertex_iter():
        vstar = vstar_lookup[v0]
        for h in f.halfedge_iter():
            if h.rev.nex.dest is vstar:
                h = h.nex
                vs.append(h.orig)
                vf0_lookup[(h.orig, f)] = (v0, f0)
                vstar_groups[vstar].add(h.orig)
vf_lookup = reverse_dict(vf0_lookup)
                
# changing this in-place automatically updates the positions
position_view = np.stack([v['pos'] for v in vs + vstars])
for v, p in zip(vs + vstars, position_view):
    v['pos'] = p
    
# assign and color the creases
for h in CP.halfedges:
    h[CREASE_ASSIGNMENT] = VALLEY
    
for f in fs:
    for h in f.halfedge_iter():
        h = h.rev.nex
        if h.rev.on_border():
            continue
        h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = MOUNTAIN
color_creases(CP)

    
face_coords0 = [torch.tensor(numpy.stack([v['pos'] for v in f0.vertex_iter()])).float() for f0 in f0s]

def torch_rot_mat(angles):
    s, c = torch.sin(angles), torch.cos(angles)
    return torch.stack([c, -s, s, c], axis=-1).view(-1, 2, 2)

class TwistrotateFaces(torch.nn.Module):
    def __init__(self, n_faces, initial_angles, initial_scale, rotation_centers, initial_star_points):
        super().__init__()
        self.scale = torch.nn.Parameter(data=torch.tensor(initial_scale).float())
        self.angles = torch.nn.Parameter(data=initial_angles.float())
        self.offsets = torch.nn.Parameter(data=torch.zeros(n_faces, 2))
        self.star_points = torch.nn.Parameter(data=initial_star_points.float())
        self.register_buffer('rotation_centers', rotation_centers.float())
        
    def forward(self, face_coords):
        # iterate over faces
        rot_matrices = torch_rot_mat(self.angles)
        result = []
        for coords, center, rot_mat, offset in zip(face_coords, self.rotation_centers, rot_matrices, self.offsets):
            result.append(((coords - center) @ rot_mat * self.scale) + center + offset)
        return torch.cat(result), self.star_points
    
    
class ConncectionLoss(torch.nn.Module):
    def __init__(self, connections, target_length_indices):
        super().__init__()
        self.connections = connections
        self.target_length_indices = target_length_indices
        
    def forward(self, face_coords, star_coords=None):
        def indices_to_lenghts(tensor, indices):
            lengths = tensor[indices]
            lengths = (lengths[:, 1] - lengths[:, 0]).pow(2).sum(-1).sqrt()
            return lengths
        lengths = indices_to_lenghts(face_coords, self.connections)
        target_lengths = indices_to_lenghts(face_coords, self.target_length_indices)
        return (lengths - target_lengths).pow(2).sum()
    
    
class StarLoss(torch.nn.Module):
    def __init__(self, star_groups):
        super().__init__()
        self.star_groups = star_groups
        
    def forward(self, face_coords, star_coords):
        loss = 0
        for group, center in zip(self.star_groups, star_coords):
            lengths = (face_coords[group] - center[None]).pow(2).sum(-1).sqrt()
            mean_length = lengths.mean()
            loss += (lengths - mean_length).pow(2).sum()
        return loss
            
class SumLoss(torch.nn.Module):
    def __init__(self, losses, weights):
        super().__init__()
        self.losses = losses
        self.weights = weights
        
    def forward(self, *args):
        return sum([w * l(*args) for w, l in zip(self.weights, self.losses)])
    
def edgelength(h):
    return np.linalg.norm(h.orig['pos'] - h.dest['pos'])


def print_metric():
    CP.recompute_lengths_and_angles()
    relative_connection_length_errors = []
    absolute_connection_length_errors = []
    angle_errors = []
    for f0 in f0s:
        for h0 in f0.halfedge_iter():
            if h0.rev.on_border():
                continue
            v, f = vf_lookup[h0.orig, f0]
            h = next(h for h in f.halfedge_iter() if h.orig is v).rev
            crease_to_star = h.nex.rev # red crease
            connection = crease_to_star.nex.rev

            # check connection length
            length = edgelength(connection)
            target_length = edgelength(h)
            absolute_connection_length_errors.append(length - target_length)
            relative_connection_length_errors.append((length - target_length) / target_length)

            # check angles which should be equal
            angle_errors.append(h['in_angle'] - crease_to_star['in_angle'])
    text = f'Maximum relative error in connection length is {np.max(np.abs(relative_connection_length_errors)):4.2%}.\n' \
           f'Maximum mismatch between angles is {np.max(np.abs(angle_errors)) * 180/np.pi:4.2f}°.'
    print(text)
    return text
    

# diagonal connections between faces of the flagstone
connections = []
target_length_indices = []
for h in G.halfedges_representing_edges():
    if h.on_border() or h.rev.on_border():
        continue
    va = vf_lookup[(h.orig, h.face)][0]
    vb = vf_lookup[(h.dest, h.rev.face)][0]
    connections.append([vs.index(va), vs.index(vb)])
    target_length_indices.append([vs.index(va), vs.index(vf_lookup[(h.dest, h.face)][0])])
    
connections = torch.LongTensor(connections)
target_length_indices = torch.LongTensor(target_length_indices)

# stars: connections to flagstone vertices should have constant length for each star
star_group_indices = []
for vstar in vstars:
    group = vstar_groups[vstar]
    star_group_indices.append([vs.index(v) for v in group])    
    
model = TwistrotateFaces(
    len(fs), 
    initial_angles = torch.ones(len(fs)) * 0, 
    initial_scale = 0.4, #0.75,
    rotation_centers = torch.from_numpy(np.stack([f.midpoint() for f in f0s])),
    initial_star_points= torch.from_numpy(np.stack([v['pos'] for v in vstars]))
)
criterion = SumLoss(
    [ConncectionLoss(connections, target_length_indices), StarLoss(star_group_indices)],
    [1, 1]
)


# optimizer = torch.optim.SGD([p for p in model.parameters()], lr=1e-6)
# works well for poincare disks
optimizer = torch.optim.SGD([
    {'params': model.offsets, 'lr': 1e-3},
    {'params': model.angles, 'lr': 1e-2},
    {'params': model.star_points, 'lr': 1e-2}
], lr=0, momentum=0.0)

# vs_positions[:] = asnumpy(torch.cat(model(face_coords0)) 
# CP.show(**render_settings)

for g in optimizer.param_groups:
    g['lr'] *= 1 #0.1
            
face_coords, star_coords = model(face_coords0)
position_view[:] = asnumpy(torch.cat([face_coords, star_coords]))    
print_metric()
CP.show(**render_settings)

training_curve = []
try:
    for i in tqdm(range(10000)):
        optimizer.zero_grad()
        face_coords, star_coords = model(face_coords0)
        loss = criterion(face_coords, star_coords)
        loss.backward()
        optimizer.step()
        training_curve.append(loss.item())

        if loss < 1e-9:
            print(f'converged at step {i}, loss={loss}')
            break

        if i == 150:
            for g in optimizer.param_groups:
                g['lr'] *= 10
                g['momentum'] = 0.9
#             optimizer.param_groups[0]['lr'] *= 0.1

        if i==200 or i%1000 == 0:
            print(model.scale)
            print(f'Iteration {i}: loss = {loss.item():4.2}')
            position_view[:] = asnumpy(torch.cat([face_coords, star_coords]))    
            print_metric()
            CP.show(**render_settings)
            if i > 0:
                plt.plot(training_curve)
                plt.yscale('log')
                plt.xlabel('step')
                plt.ylabel('loss')
                plt.grid()
                plt.show();
except Exception as e:
    print(f'Error: {e}')
finally:
    plt.plot(training_curve)
    plt.yscale('log')
    plt.grid()
    plt.show();

    position_view[:] = asnumpy(torch.cat([face_coords, star_coords]))    
    metric_info = print_metric()
    CP.delete_subset([v for v in CP.vertices if v.on_border() and v.order()==2])

    CP.show(**render_settings_big)
# n_faces = len(G.faces)
# initial_rot = torch.tensor()

In [ ]:
# extend the border of the CP

from eucare.base import angle_to_axis

def translation_mat(t):
    m = np.eye(3)
    m[:2, 2] = t
    return m

def rotation_mat(alpha):
    s, c = np.sin(alpha), np.cos(alpha)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def mirror_mat(line):
    # translate to origin
    t1 = translation_mat(-line[0])
    t2 = translation_mat(line[0])
    
    # rotate second point to x-axis
    angle = angle_to_axis(line[1] - line[0])
    r1 = rotation_mat(-angle)
    r2 = rotation_mat(angle)
    
    # mirror along x-axis
    mx = np.array([[1, 0, 0], [0, -1, 0], [0, 0, 1]], dtype=np.float64)
    
    # put it together
    return t2 @ r2 @ mx @ r1 @ t1

def apply_affine(m, v):
    return (m @ np.concatenate([v, [1]]))[:2]

CP_cleaned = CP.copy()
for h in CP_cleaned.border_edges():
    if h.rev.pre[CREASE_ASSIGNMENT] == MOUNTAIN:
        continue
    h['color_key'] = h.rev['color_key'] = (0, 1, 0)
    v = h.rev.nex.dest
    pos = apply_affine(mirror_mat([h.orig['pos'], h.dest['pos']]), v['pos'])
    h_new, _ = CP_cleaned.subdivide_face(h.rev.face, h.orig, h.dest)
    h_new[CREASE_ASSIGNMENT] = h_new.rev[CREASE_ASSIGNMENT] = MOUNTAIN
    h_new_2, v_new = CP_cleaned.subdivide_edge(h, pos=pos)
    CP_cleaned.subdivide_face(None, h_new_2.nex.dest, v_new)
color_creases(CP_cleaned)
CP_cleaned.check_consistency()
CP_cleaned.show(**render_settings_big)

In [ ]:
from eucare.utils import print_attribute_info

print_attribute_info(CP_cleaned)

In [ ]:
G = CP_cleaned.copy()

subdivisions = 15

if not isinstance(subdivisions, int) or subdivisions < 2:
    raise ValueError

for v in G.vertices:
    if 'pre_conway' in v:
        v['color_key'] = (0, 1, 0)
G.show()

# set fold angles of blue folds
for v in G.vertices:
    if 'pre_conway' not in v:
        continue
    for h in v.outgoing_iter():
        if h['color_key'] != (0, 0, 1):
            continue
        angle = np.pi - h.nex.rev['in_angle']
        opacity = angle / np.pi
        assert 0 < opacity < 1, f'{opacity}'
        h['color_key'] = h.rev['color_key'] = (0, 0, 1, opacity)

def opposite_triangle_vertex(h):
    if h.on_border():
        return None
    assert h.face.order() == 3, f'{h.face.order()}'
    return h.nex.dest

to_subdivide = set()
for h in G.halfedges:
    v1 = opposite_triangle_vertex(h)
    v2 = opposite_triangle_vertex(h.rev)
    if v1 is not None and v2 is not None and 'pre_conway' in v1 and 'pre_conway' in v2:
        h['color_key'] = (1, 1, 0)
        to_subdivide.add(h.pre)
    
while to_subdivide:
    h = to_subdivide.pop()
    to_subdivide.discard(h.rev)
    v1 = opposite_triangle_vertex(h)
    v2 = opposite_triangle_vertex(h.rev)
    
    # get subdivision points
    angle0 = ec.base.angle_to_axis(h.dest['pos'] - v1['pos'])
    angle1 = ec.base.angle_to_axis(h.orig['pos'] - v1['pos'])
    delta_angle = (angle0-angle1) % (2*np.pi)
    pts = []
    for t in np.linspace(0, 1, subdivisions, endpoint=False)[1:]:
        angle = angle0 - delta_angle * t
        pts.append(ec.base.line_intersection(
            [h.orig['pos'], h.dest['pos']],
            [v1['pos'], v1['pos'] + ec.base.unit_vector(angle)]
        ))
    pts = np.stack(pts)
#     t = np.linspace(0, 1, subdivisions, endpoint=False)[1:]
#     pts = t[:, None] * h.orig['pos'][None] + (1-t)[:, None] * h.dest['pos'][None]
    
    
    # subdivide the halfedge
    vs = []
    for pos in pts:
        vs.append(G.subdivide_edge(h, pos=pos)[1])
        
    # subdivide the first face
    for vi in vs:
        G.subdivide_face(h.face, v1, vi, color_key=(1, 1, 0))
        
    # if h is not on the border, subdivide the second face
    if not h.rev.on_border():
        for vi in vs:
            G.subdivide_face(h.rev.face, vi, v2, color_key=(1, 1, 0))
            

ps, vs = G.get_position_view()

# ps[:] *= 100
        
        
G.show(render_vertices=False, render_faces=False, line_width=0.01)

In [ ]:
2*np.pi/8/(180)

In [ ]:
# clean up the boundary: delete centers of 'twists' with only two ridges

h = next(h for h in CP.border_edges() if h.dest in vstars)
v0 = h.orig
border_poly = []
while True: # walk around boundary
    if h.dest in vstars and h.dest.order() == 4: # only two ridges
        border_poly.append(h.nex.rev.nex.dest['pos'])
    else:
        border_poly.append(h.dest['pos'])
    if h.dest == v0:
        break
    h = h.nex
border_poly = np.stack(border_poly)
print(border_poly.shape)
CP_cleaned = CP.copy()
cut_out_poly(CP_cleaned, border_poly)
color_creases(CP_cleaned, color_border=False)
CP_cleaned.show(**render_settings_big)

In [ ]:
# for folding, cut out centers of twists
CP_for_folding, (v_map_folding, _, f_map_folding) = CP.copy(return_mappings=True)
v_map_folding_rev = reverse_dict(v_map_folding)

def edge_midpoint(h): #TODO: replace with member function of edge
    return np.mean([v['pos'] for v in (h.orig, h.dest)], axis=0)

polys = []
for vstar, group in tqdm(vstar_groups.items()):
    vstar = v_map_folding[vstar]
    group = [v_map_folding[v] for v in group]
    if vstar not in CP_for_folding.vertices: # if the border was already cleaned up, this might happen
        continue
    
    # construct polygon
    if vstar.on_border():
        h = next(h1 for h1 in vstar.incoming_iter() if h1.on_border())
        h = h.rev
        poly = [h.orig['pos'], h.dest['pos']]
        while True:
            h = h.pre.rev
            if h.on_border():
                break
            h = h.pre.rev
            poly.append(h.dest['pos'])
        poly.append(edge_midpoint(h))
        poly = np.stack(poly)
    else:
        poly = np.stack([v['pos'] for v in vstar.vertex_iter() if v in group])
    polys.append(poly)

# finally, delete all the diagonal folds and assign mountain/valley

to_delete = set()
for f in fs:
    f = f_map_folding[f]
    for h in f.halfedge_iter():
        h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = MOUNTAIN
        h = h.rev.nex
        if h.rev.on_border():
            continue
        h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = VALLEY
        h = h.rev.nex
        to_delete.add(h)
color_creases(CP_for_folding)
CP_for_folding.delete_subset(to_delete)

for poly in polys:
    poly = inset_poly(poly, -1e-3)
    cut_out_poly(CP_for_folding, poly, delete_outside=True)
CP_for_folding.show(line_width=0.02)
CP_for_folding.recompute_lengths_and_angles()
wireframe = CP_for_folding.copy()
fold_wireframe(wireframe)
for f in wireframe.faces:
    if 'color_key' in f:
        del f['color_key']
wireframe.show(render_edges=False, render_vertices=False)

for poly in polys:
    poly = inset_poly(poly, -0.05)
    cut_out_poly(CP_for_folding, poly, delete_outside=True)
CP_for_folding.recompute_lengths_and_angles()

try:
    raise RuntimeError("Did not even attempt folding")
#     result = fold_complete(CP_for_folding.copy(), overlap_eps=1e-6, area_eps=0)
except Exception as e:
    print(f'Error in folding {e}')
    result = dict(CP=CP_cleaned, folded_state=wireframe, folded_view_top=G)
    
# render_settings['render_faces'] = False
# result['CP'].show(**render_settings)
# result['folded_view_top'].show(**render_settings)
# result['folded_view_bottom'].show(**render_settings)

In [ ]:
from eucare.overlap import save_results
import os

name = 'mcqueens_0'
# bbox = (25, 20)
# bbox = (50, 37)

# bbox = (35, 30)
# bbox = (65, 48)
bbox = (95, 58)

base_path = 'nice_images/alternating_flagstones'

result['folded_state'] = wireframe
result['CP'] = CP_cleaned

result_path = os.path.join(base_path, name)
assert not os.path.exists(result_path), f'"{result_path}" already exists'
save_results(result, result_path, bbox=bbox, extra_info=metric_info)

### TODO make 90° creases to simulate in origami simulator

## IDEA: glue the kised soccer ball into the center of red ring